In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

def scrape_listeners_page(page_num):
    if page_num == 1:
        url = "https://kworb.net/spotify/listeners.html"
    else:
        url = f"https://kworb.net/spotify/listeners{page_num}.html"

    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.content, "lxml")
    tabla = soup.find("table")

    if tabla is None:
        print(f"  No se encontró tabla en página {page_num}")
        return pd.DataFrame()

    filas = tabla.find_all("tr")[1:]
    resultados = []
    for fila in filas:
        celdas = fila.find_all("td")
        if len(celdas) < 5:
            continue

        enlace = fila.find("a")
        spotify_id = None
        if enlace and "href" in enlace.attrs:
            match = re.search(r"artist/([^_]+)_songs\.html", enlace["href"])
            if match:
                spotify_id = match.group(1)

        # peak_pos solo disponible en página 1 (6 columnas)
        if len(celdas) == 6:
            resultados.append({
                "ranking":        celdas[0].text.strip(),
                "artista":        celdas[1].text.strip(),
                "spotify_id":     spotify_id,
                "listeners":      celdas[2].text.strip().replace(",", ""),
                "daily_change":   celdas[3].text.strip().replace(",", ""),
                "peak_pos":       celdas[4].text.strip(),
                "peak_listeners": celdas[5].text.strip().replace(",", ""),
            })
        else:
            resultados.append({
                "ranking":        celdas[0].text.strip(),
                "artista":        celdas[1].text.strip(),
                "spotify_id":     spotify_id,
                "listeners":      celdas[2].text.strip().replace(",", ""),
                "daily_change":   celdas[3].text.strip().replace(",", ""),
                "peak_pos":       None,
                "peak_listeners": celdas[4].text.strip().replace(",", ""),
            })
    return pd.DataFrame(resultados)

# --- Ejecución ---
dfs = []
for i in range(1, 11):
    print(f"Scrapeando página {i}...")
    df_page = scrape_listeners_page(i)
    print(f"  Filas: {len(df_page)}")
    dfs.append(df_page)
    time.sleep(2)

df_listeners = pd.concat(dfs, ignore_index=True)
df_listeners.to_csv("kworb_listeners.csv", index=False)
print(f"Total artistas: {len(df_listeners)}")
print(f"\nValores nulos por columna:")
print(df_listeners.isnull().sum())
df_listeners.head()

Scrapeando página 1...
  Filas: 2500
Scrapeando página 2...
  Filas: 2500
Scrapeando página 3...
  Filas: 2500
Scrapeando página 4...
  Filas: 2500
Scrapeando página 5...
  Filas: 2500
Scrapeando página 6...
  Filas: 2500
Scrapeando página 7...
  Filas: 2500
Scrapeando página 8...
  Filas: 2500
Scrapeando página 9...
  Filas: 2500
Scrapeando página 10...
  Filas: 2384
Total artistas: 24884

Valores nulos por columna:
ranking               0
artista               0
spotify_id            0
listeners             0
daily_change          0
peak_pos          22384
peak_listeners        0
dtype: int64


,ranking,artista,spotify_id,listeners,daily_change,peak_pos,peak_listeners
0,1,Bruno Mars,0du5cEVh5yTK9QJze8zA0C,135190715,53657,1,151079821
1,2,The Weeknd,1Xyo4u8uXC1ZmMpatF05PJ,116058365,80638,1,126192069
2,3,Rihanna,5pKCCKE2ajJHZ9KAiaK11H,108677292,73880,2,108677292
3,4,Bad Bunny,4q3ewBCX7sLwd24euuV69X,105319095,-301986,2,123934765
4,5,Taylor Swift,06HL4z0CvFAxyc27GXpf02,102038503,-36940,1,116229071


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

def scrape_artists():
    url = "https://kworb.net/spotify/artists.html"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.content, "lxml")
    tabla = soup.find("table")

    if tabla is None:
        print("No se encontró tabla")
        return pd.DataFrame()

    filas = tabla.find_all("tr")[1:]
    resultados = []
    for i, fila in enumerate(filas):
        celdas = fila.find_all("td")
        if len(celdas) < 6:
            continue

        enlace = fila.find("a")
        spotify_id = None
        if enlace and "href" in enlace.attrs:
            match = re.search(r"/spotify/artist/([^_]+)_songs\.html", enlace["href"])
            if match:
                spotify_id = match.group(1)

        resultados.append({
            "ranking":    i + 1,
            "artista":    celdas[0].text.strip(),
            "spotify_id": spotify_id,
            "streams":    celdas[1].text.strip().replace(",", ""),
            "daily":      celdas[2].text.strip().replace(",", ""),
            "as_lead":    celdas[3].text.strip().replace(",", ""),
            "solo":       celdas[4].text.strip().replace(",", ""),
            "as_feature": celdas[5].text.strip().replace(",", ""),
        })
    return pd.DataFrame(resultados)

# --- Ejecución ---
df_artists = scrape_artists()
df_artists.to_csv("kworb_artists_careful.csv", index=False)
print(f"Total artistas: {len(df_artists)}")
print(f"\nValores nulos por columna:")
print(df_artists.isnull().sum())
df_artists.head()

Total artistas: 3000

Valores nulos por columna:
ranking       0
artista       0
spotify_id    0
streams       0
daily         0
as_lead       0
solo          0
as_feature    0
dtype: int64


,ranking,artista,spotify_id,streams,daily,as_lead,solo,as_feature
0,1,Drake,3TVXtAsR1Inumwj472S9r4,130980.0,46.974,89592.4,49837.8,41387.7
1,2,Taylor Swift,06HL4z0CvFAxyc27GXpf02,123426.1,43.090,119680.0,109401.9,3746.1
2,3,Bad Bunny,4q3ewBCX7sLwd24euuV69X,120238.1,56.913,78164.1,46145.5,42074.0
3,4,The Weeknd,1Xyo4u8uXC1ZmMpatF05PJ,92416.3,31.420,74463.8,49386.5,17952.5
4,5,Justin Bieber,1uNFoZAHBGtllmzznpCI3s,73063.8,56.333,44225.6,27247.6,28838.2


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

def scrape_totals(url, tipo):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.content, "lxml")
    tabla = soup.find("table")

    if tabla is None:
        print(f"No se encontró tabla en {url}")
        return pd.DataFrame()

    filas = tabla.find_all("tr")[1:]
    resultados = []
    for fila in filas:
        celdas = fila.find_all("td")
        if len(celdas) < 6:
            continue

        # Extraer artista y título del primer campo
        texto = celdas[0].text.strip()
        enlace = fila.find("a")
        spotify_id = None
        if enlace and "href" in enlace.attrs:
            match = re.search(r"artist/([^/]+)/", enlace["href"])
            if match:
                spotify_id = match.group(1)

        # Separar artista y título por " - "
        if " - " in texto:
            artista, titulo = texto.split(" - ", 1)
        else:
            artista, titulo = texto, None

        # Extraer peak y multiplicador del campo Pk(x?)
        pk_texto = celdas[3].text.strip()
        pk_match = re.match(r"(\d+)(?:\(x(\d+)\))?", pk_texto)
        pk_pos = pk_match.group(1) if pk_match else None
        pk_veces = pk_match.group(2) if pk_match and pk_match.group(2) else "1"

        if tipo == "daily":
            resultados.append({
                "artista":     artista.strip(),
                "titulo":      titulo.strip() if titulo else None,
                "spotify_id":  spotify_id,
                "dias":        celdas[1].text.strip().replace(",", ""),
                "dias_top10":  celdas[2].text.strip().replace(",", ""),
                "peak_pos":    pk_pos,
                "peak_veces":  pk_veces,
                "peak_streams": celdas[4].text.strip().replace(",", ""),
                "total_streams": celdas[5].text.strip().replace(",", ""),
            })
        else:  # weekly
            resultados.append({
                "artista":      artista.strip(),
                "titulo":       titulo.strip() if titulo else None,
                "spotify_id":   spotify_id,
                "semanas":      celdas[1].text.strip().replace(",", ""),
                "semanas_top10": celdas[2].text.strip().replace(",", ""),
                "peak_pos":     pk_pos,
                "peak_veces":   pk_veces,
                "peak_streams": celdas[4].text.strip().replace(",", ""),
                "total_streams": celdas[5].text.strip().replace(",", ""),
            })

    return pd.DataFrame(resultados)

# --- Ejecución ---
df_daily_totals = scrape_totals(
    "https://kworb.net/spotify/country/es_daily_totals.html",
    tipo="daily"
)
df_daily_totals.to_csv("kworb_spain_daily_totals.csv", index=False)
print(f"Daily totals: {len(df_daily_totals)} filas")
print(df_daily_totals.isnull().sum())

df_weekly_totals = scrape_totals(
    "https://kworb.net/spotify/country/es_weekly_totals.html",
    tipo="weekly"
)
df_weekly_totals.to_csv("kworb_spain_weekly_totals.csv", index=False)
print(f"\nWeekly totals: {len(df_weekly_totals)} filas")
print(df_weekly_totals.isnull().sum())

Daily totals: 10760 filas
artista              0
titulo               7
spotify_id       10760
dias                 0
dias_top10           0
peak_pos             0
peak_veces           0
peak_streams         0
total_streams        0
dtype: int64

Weekly totals: 7182 filas
artista             0
titulo              1
spotify_id       7182
semanas             0
semanas_top10       0
peak_pos            0
peak_veces          0
peak_streams        0
total_streams       0
dtype: int64
